In [ ]:
import pandas as pd
import glob
import logging
import os
import gc
import numpy as np
from frameaxis.core import random_subset_null

In [ ]:
logging.basicConfig(level=logging.DEBUG,
                    format='%(asctime)s %(name)-12s %(levelname)-8s %(message)s',
                    datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger(__name__)

# Null model: random subsets of the whole table

For each aspect x sentiment cell, draw `N` subsets of the same size as the cell, without replacement, from the whole table, and save their column means. The seed makes reruns identical. Samples are saved as float32 `.npy` (about 65 MB per cell for N = 10,000).

In [ ]:
def bootstrapping(df, mode, COLUMNS):
    A = df[COLUMNS].values
    for aspect in df['aspect'].unique():
        for sentiment in ['positive', 'negative']:
            df_r_count = df.query('aspect == @aspect and sentiment == @sentiment').shape[0]
            logger.info("{}-{}-{}".format(mode, aspect, sentiment))
            null = random_subset_null(A, df_r_count, N, SEED)
            np.save("{}/bootstrap_{}_{}.npy".format(mode, aspect.replace("/", "-"), sentiment), null.astype(np.float32))

In [ ]:
N = 10000
SEED = 2021

In [ ]:
mode = "average"
logger.info(mode)
os.makedirs(mode, exist_ok=True)

df = pd.read_csv("big_table_by_{}.tsv".format(mode), sep="\t").dropna()
COLUMNS = [c for c in df.columns if '(' in c]
len(COLUMNS)

In [ ]:
bootstrapping(df, mode, COLUMNS)

In [ ]:
mode = "second_moment"
logger.info(mode)
os.makedirs(mode, exist_ok=True)

df = pd.read_csv("big_table_by_{}_with_corpus_mean.tsv".format(mode), sep="\t").dropna()
COLUMNS = [c for c in df.columns if '(' in c]
len(COLUMNS)

In [ ]:
bootstrapping(df, mode, COLUMNS)